# Klasifikasi DemogPairs Menggunakan ViT (Emosi) & Gaussian Naive Bayes

In [1]:
import numpy as np
import utils as u
import joblib
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import StratifiedKFold, ParameterGrid
from imblearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from tqdm import tqdm

joblib.parallel_backend('threading')

## Load Dataset

In [3]:
data = u.load_demogpairs()
pd.DataFrame(data)

## Load Fitur

In [5]:
features = joblib.load('features/demogpairs_vit-emotion.pkl')
print('Jumlah fitur per gambar:', np.array(features[list(features.keys())[0]]).shape[0])

Jumlah fitur per gambar: 768


## Split Data

In [7]:
X = np.array([features[d['image_path']] for d in data])
y = np.array([d['label_idx'] for d in data])
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)
print((len(X_train), len(X_test)))

(8640, 2160)


## Kombinasi Parameter

In [9]:
var_smoothing_values = np.logspace(-9, 2, 40)  # dari 1e-9 sampai 1e2, 40 nilai

grid_params = [
    {
        'scaler': [None, MinMaxScaler()],
        'pca': [None, PCA(n_components=0.5), PCA(n_components=0.75)],
        
        'classifier': [GaussianNB()],
        'classifier__var_smoothing': var_smoothing_values
    },
]

pipeline = Pipeline(steps=[
    ('scaler', None),
    ('pca', None),
    ('classifier', None)
])

skv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = {
    'accuracy': 'accuracy', 
    'f1': 'f1_macro', 
    'precision': 'precision_macro', 
    'recall': 'recall_macro',
    'roc_auc_ovr': 'roc_auc_ovr'
}

grid_models = {}
for params in grid_params:
    key = str(params['classifier'][0]).split('(')[0]
    grid_models[key] = GridSearchCV(
        estimator=pipeline,
        param_grid=params,
        cv=skv, refit='accuracy',
        scoring=scoring, n_jobs=int(joblib.cpu_count() * 0.6),
        verbose=1, error_score='raise',
        return_train_score=True
    )
    print(f'{key}: {len(ParameterGrid(params))} kombinasi')

GaussianNB: 240 kombinasi


## Klasifikasi

In [11]:
evaluation_results, fold_results = u.evaluate_models(
    grid_models,
    X_train, y_train,
    X_test, y_test,
    model_prefix='models/clf_demogpairs_gnb_vit-emotion_',
    results_path='results/demogpairs_gnb_vit-emotion_'
)

sorted_results = pd.DataFrame(evaluation_results).sort_values(by='test_accuracy', ascending=False).to_dict('records')
u.html_br()
_dtable = u.display_table(sorted_results)

Evaluating: GaussianNB

##### Best Parameters
{'classifier': 'GaussianNB', 'classifier__var_smoothing': np.float64(0.00307029062975785), 'pca': 'PCA', 'scaler': None}

##### Test Results
Accuracy  : 0.7337962962962963
Precision : 0.7386856751531776
Recall    : 0.7337962962962964
F1 Score  : 0.7329054843249007
              precision    recall  f1-score   support

           0       0.83      0.69      0.76       360
           1       0.69      0.81      0.75       360
           2       0.67      0.68      0.67       360
           3       0.73      0.84      0.78       360
           4       0.74      0.75      0.74       360
           5       0.77      0.64      0.69       360

    accuracy                           0.73      2160
   macro avg       0.74      0.73      0.73      2160
weighted avg       0.74      0.73      0.73      2160


Class
    Accuracy
    Precision
    Recall
    F1-Score
  
  
    Black_Males
    0.9254629629629629
    0.8305647840531561
    0.69444444444444

In [12]:
model, training_time = u.load_object('models/clf_demogpairs_gnb_vit-emotion_GaussianNB.pkl')
u.h(5, 'Waktu Pelatihan (Jobs)')
u.seconds_to_time(round(training_time))


##### Waktu Pelatihan (Jobs)


In [13]:
u.h(5, 'Waktu Pelatihan')
times = [fr['Train Time Mean'] * 5 for fr in fold_results]
u.seconds_to_time(round(np.sum(times) + model.refit_time_))


##### Waktu Pelatihan


In [14]:
_dtable = u.display_table(fold_results, n_items=[4, 4], column_widths=['5%', '45%', '5%', '5%', '5%', '5%', '5%', '5%', '5%', '5%', '5%', '5%'])

No
    Params
    Fold 1
    Fold 2
    Fold 3
    Fold 4
    Fold 5
    Accuracy Mean
    F1 Score Mean
    Precision Mean
    Recall Mean
    Train Time Mean
  
  
    1
    {'classifier': 'GaussianNB', 'classifier__var_smoothing': np.float64(0.00307029062975785), 'pca': 'PCA', 'scaler': None}
    0.7222
    0.7355
    0.7245
    0.7251
    0.7483
    0.7311
    0.7306
    0.738
    0.7311
    1.948
  
  
    2
    {'classifier': 'GaussianNB', 'classifier__var_smoothing': np.float64(0.0058780160722749115), 'pca': 'PCA', 'scaler': None}
    0.7251
    0.735
    0.7257
    0.7199
    0.7471
    0.7306
    0.7301
    0.7379
    0.7306
    1.9386
  
  
    3
    {'classifier': 'GaussianNB', 'classifier__var_smoothing': np.float64(0.001603718743751331), 'pca': 'PCA', 'scaler': None}
    0.7193
    0.7344
    0.7263
    0.7251
    0.7471
    0.7304
    0.73
    0.7369
    0.7304
    2.0376
  
  
    4
    {'classifier': 'GaussianNB', 'classifier__var_smoothing': np.float64(0.00022854638641